In [1]:
from typing import List, TypedDict, Annotated
import re
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# FIX 1: HuggingFaceEndpoint & ChatHuggingFace import added

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()


d:\Coding\Genarative-AI\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
from langchain_groq import ChatGroq
import os
from langchain_core.output_parsers import StrOutputParser

In [2]:
docs = PyPDFLoader("./documents/book1.pdf").load()

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=100)
split_docs=splitter.split_documents(docs)
print(len(split_docs))

2433


In [7]:

model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [8]:
vector_store = FAISS.from_documents(split_docs,embed_model)

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",   
)

In [ ]:
# retriever sentence striper funtion
def decompose_to_sentence(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()

    
    raw_sentences = re.split(r"(?<=[.!?])\s+", text)

    
    chunks = []
    current_chunk = ""

    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        
        if len(current_chunk) + len(s) + 1 > 400 and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = s  
        else:
            current_chunk = (current_chunk + " " + s).strip()

   
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


In [40]:
class State(TypedDict):
    question:str
    answer:str
    docs:list[Document]

    all_node_strips:list[list]
    all_strips:list[str]
    kept_strips:list[str]
    refined_context:str

In [48]:
def retrieve(state):
    q = state["question"]
    return {'docs':retriever.invoke(q)}

In [49]:
filter_prompt = ChatPromptTemplate(
    [
        ('system',"You are a strict relevance filter. \n"
         "Does the sentence directly help answer the question? \n"
         "Reply with ONLY one word: YES or NO. Nothing else."),
        ('human','question: {question}\n\nsentence:{sentence}')
    ]
)

filter_chain = filter_prompt | model |StrOutputParser() | (lambda x:"YES" in x.strip().upper())

In [54]:
def refine(state:State)->State:
    q = state['question']
    docs = state['docs']
    all_node_strips = []
    all_inputs_strips_withQ = []
    all_strips = []
    kept_strips = []
    for doc in docs:
        strips = decompose_to_sentence(doc.page_content)
        all_node_strips.extend(strips)


        for s in strips:
            all_inputs_strips_withQ.append({'question':q,'sentence':s})
            all_strips.append(s)
    
    all_out = filter_chain.batch(all_inputs_strips_withQ)

    for s,r in zip(all_strips,all_out):
        if r:
            kept_strips.append(s)

    final_context = '\n\n'.join(kept_strips)

    return{
        'all_strips':all_inputs_strips_withQ,
        'kept_strips':kept_strips,
        'refined_context':final_context,
        'all_node_strips':all_node_strips
    }



In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"Answer only from the context. If not in contex, say you don't know",),
        ('human', "Question : {question}\n\nContext:\n{context}")
    ]
)


def generate(state: State) -> State:
    generate_chain = answer_prompt | llm | StrOutputParser()  
    out = generate_chain.invoke({
        "question": state["question"], 
        "context": state['refined_context']
    })
    return {"answer": out} 

In [ ]:

g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "refine")
g.add_edge("refine", "generate")
g.add_edge("generate", END)

app = g.compile()
print("Graph compiled successfully!")


Graph compiled successfully!


In [67]:
res = app.invoke({
    "question": "What is Neural Networks?",

})

print("Answer:", res["answer"])
print("Kept sentences count:", len(res['kept_strips']))


Answer: Based on the provided context, to learn the parameters w of the neural network using maximum likelihood, we can follow these steps:

1. Define the error function or likelihood function, which measures the goodness of fit of the model to the data. This function is typically differentiable with respect to the network parameters w.
2. Evaluate the derivatives of the error function with respect to the weights w in the first stage. This is necessary because most training algorithms involve an iterative procedure for minimization of the error function.
3. Use an optimization algorithm to minimize the error function by adjusting the weights w in a sequence of steps. The optimization algorithm will use the evaluated derivatives to determine the direction of the updates.
4. Repeat the process of evaluating the derivatives and updating the weights until convergence or a stopping criterion is reached.

However, the specific details of the maximum likelihood estimation process are not full

In [70]:
r = res['kept_strips']
l = len(r)
print(r)
print(l)


['The term ‘neural network’ has its origins in attempts to ﬁnd mathematical rep- resentations of information processing in biological systems (McCulloch and Pitts, 1943; Widrow and Hoff, 1960; Rosenblatt, 1962; Rumelhart et al., 1986). Indeed,', 'for this reason the neural network is also known as the multilayer perceptron,o r MLP. A key difference compared to the perceptron, however, is that the neural net- work uses continuous sigmoidal nonlinearities in the hidden units, whereas the per- ceptron uses step-function nonlinearities.', 'This means that the neural network func- tion is differentiable with respect to the network parameters, and this property will play a central role in network training. If the activation functions of all the hidden units in a network are taken to be linear, then for any such network we can always ﬁnd an equivalent network without hidden units.', 'Most training algorithms involve an iterative procedure for minimization of an error function, with adjustment